# Lab 2: Measurement & Indexes — Deflating History
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 8 min Part 0 + 45 min Core + 10 min optional API section

---

**Learning Objectives:**
- Pull real FRED time series into pandas — first with no credential at all, then with your own API key
- Read and use a reusable `deflate_series()` function
- Plot the fifty-year divergence between nominal and real wages
- Deflate Big Mac prices to separate inflation from a real price change

**Dataset:** FRED (CPIAUCSL, AHETPI) and The Economist's Big Mac Index

**How this lab works:** Parts 0-3 are GUIDED: run each cell as it is, then answer the
questions. Part 4 has three blanks for you to fill in. Part 5 is optional.

**Prerequisites:** Part 0 of this lab (below), plus Lab 1 (pandas on real data), Chapter 2
reading (CPI, deflation, Goodhart's Law)

**AI in the manual parts:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion while you work on those parts. AI-assisted coding is allowed only where this lab says so.

---

> ### What you write, and what you run
>
> **You write:** three blanks in Step 8 (Part 4), one short expression each, about three lines of code in all. In text cells: the answers to the interpretation questions, one sentence after the Step 9 chart, and the three answers after the AI Expansion. Your README (Part 6), drafted with the prompt given and checked against your own numbers.
>
> **You paste:** the explorer code an AI writes for you, in the AI Expansion cell. Run it and check it against numbers from this lab.
>
> **You run and read:** every other code cell. They are working code to learn from. Read each one before you run it, and make sure you can say what each line does. Part 5 is optional and needs your own FRED API key.
>
> **For Problem Set 1:** PS1 pulls FRED series and rebases them to 2019 = 100. Read `load_fred()` in Setup and Steps 1 and 5, which pull a series. From `deflate_series()` in Step 3, reuse only the base-year average line; the rest of the function deflates, which PS1 does not ask for.

## Before you start: FRED, and what an API key is

**FRED** — Federal Reserve Economic Data, run by the St. Louis Fed — publishes over
800,000 economic time series for free. This lab uses two of them: the Consumer Price
Index (`CPIAUCSL`) and average hourly earnings (`AHETPI`).

There are two ways to get data out of FRED, and this lab shows you both.

**Route 1 — the CSV endpoint. No account, no key, nothing to sign up for.**
FRED will hand any series to `pandas.read_csv()` from a plain URL. This is what
Parts 1-4 use for every pull, so **you can complete this entire lab without a
credential.** Nothing here is blocked on paperwork.

**Route 2 — the API, with your own key.** Part 5 is optional and shows the same
data pulled through the `fredapi` package. That route needs an API key.

### What an API key actually is

An **API key** is a long string of letters and numbers that identifies *you* to a
data provider's server — a library card number for a database. The server uses it
to see who is asking, count your requests, and cut you off if you hammer it. It is
not a password to your computer and it is not secret in the way a bank password is,
but it *is* tied to your account, so you do not publish it.

### Getting one (about two minutes)

1. Go to <https://fredaccount.stlouisfed.org/login/secure/> and click
   **Create New Account**. It is free and needs only an email address.
2. Check your email and click the confirmation link.
3. Signed in, go to <https://fredaccount.stlouisfed.org/apikeys> and click
   **Request API Key**.
4. Type anything reasonable in the description box ("ECON 3916 coursework") and
   submit.
5. Your key appears on that page immediately — a 32-character lowercase string. It
   also arrives by email. Copy it.
6. **Do not paste it into this notebook.** It goes in Colab's secret store — next section.

### Where the key goes: Colab Secrets

Colab has a built-in secret store. The key lives with your Google account, not in the
file you push, so there is nothing to remember to delete later.

1. In Colab's **left sidebar**, click the **key icon** (its tooltip says *Secrets*).
2. Click **+ Add new secret**.
3. Under **Name** type exactly `FRED_API_KEY` — capitals and the underscore, no spaces.
4. Paste your 32-character key into **Value**.
5. Switch **Notebook access** on. This is the step people miss: without it, Part 5's
   first cell stops with "No FRED API key found" even though the secret exists.

The notebook then reads it in two lines, and your key appears nowhere in the file:

```python
from google.colab import userdata
FRED_API_KEY = userdata.get("FRED_API_KEY")
```

You do this **once**. The secret is available in every notebook you open with the same
Google account, so Lab 20, Lab 21 and Problem Set 1 pick it up with no extra work.

### Three rules for every key in this course

- **Never type a key into a cell you will push.** Your lab repository is public; keys
  in public repos are scraped within minutes and FRED will revoke yours.
- **Never paste a key into a chat with an AI assistant.**
- **Never leave a key on screen** — it is captured by a screen share, a projector, and
  every frame of a recording. If that happens, revoke it and request a new one;
  FRED does both in one click at
  <https://fredaccount.stlouisfed.org/apikeys>.

## Setup

`load_fred()` below downloads one FRED series from its public CSV link, with no
API key, and returns it as a pandas Series indexed by date. Every FRED pull in
Parts 1-4 goes through it.

In [ ]:
# GUIDED — run as-is
import pandas as pd
import matplotlib.pyplot as plt


def load_fred(series_id, start="1947-01-01"):
    """Return one FRED series, from `start` to the latest month, as a Series indexed by date."""
    # id picks the series; cosd is the first date to download
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}&cosd={start}"
    frame = pd.read_csv(url)

    # Name the two columns ourselves: FRED has renamed its header before
    frame.columns = ["date", series_id]
    frame["date"] = pd.to_datetime(frame["date"])

    # errors="coerce" turns a value that is not a number (a blank month) into NaN
    frame[series_id] = pd.to_numeric(frame[series_id], errors="coerce")

    # dropna() deletes months with no value, so a NaN count afterwards is always 0: check the dates
    return frame.set_index("date")[series_id].dropna()

<!-- 3916-onramp -->
## Part 0: The Floor (GUIDED — 8 min)

Part 0 of Labs 1 through 5 teaches the slice of Python that lab needs, next to
the data that needs it. Run every cell before going on.

**This lab's slice:** lists, the `for` loop, `def`, and `.map()` with a lambda.

Part 2 hands you `deflate_series()`, a function. Functions are new, so this
Part 0 spends most of its time on one.

- A **list** is an ordered box of values, in square brackets. Python counts
  from 0, so `years[0]` is the first item and `years[-1]` the last. A slice
  `years[1:3]` includes position 1 and stops before position 3.
- A **for loop** runs its indented block once per item. The indentation is how
  Python knows where the block ends. `zip()` walks two lists side by side.
- In an f-string, `{p:.2f}` puts the value of `p` into the text with two
  decimals.
- A **function** is a named recipe. `def` starts it, the indented block is the
  body, and `return` hands a value back. The names in parentheses are
  **parameters**, filled in when you call it. `cpi_base=100` gives one a
  **default**, so the caller may leave it out.
- `.map()` runs a function on every value of a Series and returns a new
  Series. `lambda x: x * 100` is a one-line function with no name, written
  where `.map()` needs it.

In [ ]:
# GUIDED — run as-is
# Step 0: lists and the for loop

# Big Mac prices in USD, for practice only (Part 4 loads the real series)
years  = [1980, 1990, 2000, 2010, 2020]
prices = [1.60, 2.20, 2.51, 3.71, 5.71]

print("How many:      ", len(years))
print("First (idx 0): ", years[0])
print("Last  (idx -1):", years[-1])
print("Slice [1:3]:   ", years[1:3])        # positions 1 and 2, not 3

print("\n--- nominal prices ---")
for y, p in zip(years, prices):
    print(f"  {y}  ${p:.2f}")

In [ ]:
# GUIDED — run as-is
# Step 0, continued: your first function

def deflate(nominal, cpi_then, cpi_base=100):
    """Convert a nominal value into base-year dollars."""
    return nominal * cpi_base / cpi_then


# deflate_series() in Part 2 does the same thing to a whole series at once
print(f"$5.71 in 2020 (CPI 258.8), in 1980 dollars (CPI 82.4): "
      f"${deflate(5.71, 258.8, 82.4):.2f}")

In [ ]:
# GUIDED — run as-is
# Step 0, continued: .map() with a lambda

# The practice prices as a Series, labelled by year
practice = pd.Series(prices, index=years)

print("Prices in cents, via .map(lambda ...):")
# to_string() prints every row, without the dtype line at the bottom
print(practice.map(lambda x: x * 100).to_string())

## Part 1: Pulling CPI Data from FRED (GUIDED — 10 min)

FRED publishes over 800,000 economic time series for free. We start with the
Consumer Price Index.

In [ ]:
# GUIDED — run as-is
# Step 1: pull CPI for All Urban Consumers (FRED series CPIAUCSL, monthly, seasonally adjusted)

# to_frame() turns the Series into a one-column table, with the column named "cpi"
cpi = load_fred("CPIAUCSL", start="1947-01-01").to_frame(name="cpi")

print(f"CPI series: {len(cpi)} monthly observations")
# .date() shows a timestamp as a plain date
print(f"Date range: {cpi.index.min().date()} to {cpi.index.max().date()}")
# .iloc[-1] is the last row, counted by position
print(f"Latest CPI value: {cpi['cpi'].iloc[-1]:.1f}")
cpi.head()

**Expected output:** 955 monthly observations from 1947-01-01 to 2026-08-01, not
the 956 months in that span. One month has no CPI value on FRED, and `dropna()` in
`load_fred()` removes that row, so counting NaNs afterwards always gives 0. To find
a gap, look at the dates, not the NaNs. The latest CPI is above 330. FRED adds a
month every few weeks, so your count and latest value may be a little higher.

<!-- idiom-note -->
> ### New idiom — `asfreq("MS")`, and finding a gap you cannot see
>
> The cell above says one month is missing. **It does not say which**, and you cannot find
> out by looking &mdash; 955 rows all look present. `dropna()` inside `load_fred()` removed the
> empty row, so counting `NaN` afterwards returns zero and tells you nothing.
>
> `asfreq("MS")` rebuilds the index as **every Month Start from the first date to the last**,
> whether or not the data has a row for it, and writes `NaN` where there was none. The gap
> becomes visible because you asked for the calendar, not for the data.
>
> **Problem Set 1, Phase 3.1 asks you to do exactly this** for every series you pull, and to
> report the count *whatever it is* &mdash; including zero.


In [ ]:
# GUIDED — run as-is
# Step 1b: put the series on a complete monthly grid, and find the gap

# "MS" is Month Start. asfreq() does not invent data: it rebuilds the index as every
# month between the first and last date, and writes NaN wherever there was no row.
cpi_grid = cpi.asfreq("MS")

missing = cpi_grid[cpi_grid["cpi"].isna()]

print(f"rows FRED returned             : {len(cpi)}")
print(f"rows on a complete monthly grid: {len(cpi_grid)}")
print(f"missing months                 : {len(missing)}")
print(f"span                           : {cpi.index.min().date()} -> {cpi.index.max().date()}")


**Expected output:** 955 rows returned, 956 on a complete monthly grid, so **1** missing
month: the gap the cell above predicted from the row count.

**Which month it is, why it is missing, and what you will do about it** is the question
Problem Set 1, Phase 3.1 asks, for every series you pull. `missing.index` holds the dates
when you get there.

Run the same lines on a series with no gaps and the count comes back 0. Report that too:
an audit that only gets written up when it finds something is not an audit.

### Interpretation Question 1

The CPI was about 21.5 in January 1947 and is now above 330. The base period is 1982-84 = 100.

**Q:** If the CPI is currently 330, what does that number mean in plain English? Write one sentence.

*Your answer here:*



In [ ]:
# GUIDED — run as-is
# Step 2: plot the CPI level over time

# This is the LEVEL of prices, not the inflation rate: consumers live at levels
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(cpi.index, cpi["cpi"])
ax.axhline(100, color="gray", linestyle="--", label="Base period (1982-84 = 100)")
ax.set_xlabel("Year")
ax.set_ylabel("CPI (1982-84 = 100)")
ax.set_title("Prices today are more than 15 times their 1947 level")
ax.legend()
plt.show()

## Part 2: Building `deflate_series()` (GUIDED — 15 min)

`deflate_series()` does to a whole series what Part 0's `deflate()` did to one
number. You will see it again: Chapter 11 does the same deflation in SQL, and
Chapters 20 and 21 come back to these CPI and wage series.

The deflation formula: **Real Value = (Nominal Value / CPI) × Base-Year CPI**

The base-year CPI is the average of that year's monthly CPI values, so a 2020
base puts every date in average-2020 dollars.

<!-- idiom-note -->
> ### New idiom — reading a function definition, and `lambda`
>
> The cell below hands you a **function**. Problem Set 1 asks you to write
> one, `laspeyres()`. Copy this shape: `def` line, one-line docstring, body,
> `return`. For now, read `def
> deflate_series(nominal, cpi, base_year=2020):` as *"to deflate a series you
> need a nominal series, a CPI, and optionally a base year, which is 2020
> unless you say otherwise."*
>
> `lambda`, which Part 0 handed to `.map()`, is a function with no name, for
> exactly this hand-it-to-something-else case. If it needs a name, use `def`.

In [ ]:
# GUIDED — run as-is
# Step 3: the deflate_series() function

def deflate_series(nominal, cpi, base_year=2020):
    """Convert a nominal series, indexed by date, into base_year dollars."""
    # cpi.index.year == base_year is True for the base year's months; average those.
    # PS1's rebasing reuses only this line.
    base_cpi = cpi.loc[cpi.index.year == base_year].mean()

    # One table from both Series lines them up by date; dropna() keeps dates where both have a value
    aligned = pd.DataFrame({"nominal": nominal, "cpi": cpi}).dropna()

    real = (aligned["nominal"] / aligned["cpi"]) * base_cpi
    real.name = f"real_{base_year}"
    return real

In [ ]:
# GUIDED — run as-is
# Step 4: $5,000 earned in 1970, in 2020 dollars

# .loc[rows, column]: the rows where the year is 1970, from the "cpi" column
cpi_1970 = cpi.loc[cpi.index.year == 1970, "cpi"].mean()
cpi_2020 = cpi.loc[cpi.index.year == 2020, "cpi"].mean()

real_value = (5000 / cpi_1970) * cpi_2020

print(f"CPI in 1970 (annual average): {cpi_1970:.1f}")
print(f"CPI in 2020 (annual average): {cpi_2020:.1f}")
# :,.0f adds a comma every three digits and shows no decimals
print(f"$5,000 in 1970 = ${real_value:,.0f} in 2020 dollars")

A salary today is in today's dollars. To compare it with the 1970 figure, put
**both** in 2020 dollars first. Comparing a nominal 2025 salary with a
2020-dollar figure mixes two currencies.

In [ ]:
# GUIDED — run as-is
# Step 4, continued: a $55,000 salary in 2025, in the same 2020 dollars

months_2025 = cpi.loc[cpi.index.year == 2025, "cpi"]
cpi_2025 = months_2025.mean()
salary_2025_in_2020_dollars = (55000 / cpi_2025) * cpi_2020

print(f"CPI in 2025 (average of {len(months_2025)} months): {cpi_2025:.1f}")
print(f"$55,000 in 2025 = ${salary_2025_in_2020_dollars:,.0f} in 2020 dollars")

real_gain = (salary_2025_in_2020_dollars / real_value - 1) * 100
mixed_gain = (55000 / real_value - 1) * 100
print(f"\nGain, both in 2020 dollars:           {real_gain:.0f}%")
print(f"Gain, nominal 2025 over 2020 dollars: {mixed_gain:.0f}%")

**Expected output:** the 2025 average covers 11 months, not 12: one month has no
CPI value (see Step 1). \$5,000 in 1970 is about \$33,300 in 2020 dollars, and
\$55,000 in 2025 is about \$44,200. Compared in the same dollars, the 2025
salary is about 33% higher. Dividing the nominal \$55,000 by the 2020-dollar
figure gives about 65%, roughly double: that is the units error this chapter
exists to prevent.

<!-- idiom-note -->
> ### New idiom — rebasing, and why two index numbers cannot be compared as they stand
>
> `deflate_series()` above moved money into one year's dollars. **Rebasing does the same
> job for an index**: it moves an index onto one year's scale.
>
> You need it because an index number is meaningless on its own — it only means something
> relative to whatever period was set to 100. FRED's CPI is set to **1982–84 = 100**. A
> household index you build yourself will be set to whatever year you chose. Plotting the
> two together before rebasing compares a number measured from 1983 against a number
> measured from 2019, and the gap you see is mostly the difference in starting points.
>
> The whole operation is **one line**: divide every value by the base period's value, and
> multiply by 100. **Problem Set 1, Phase 2.2 asks you to do exactly this** — rebase your
> FRED series to 2019 = 100 so they sit on the same axis as your own basket.


In [ ]:
# GUIDED — run as-is
# Step 4, continued: rebasing CPI from 1982-84 = 100 onto 2019 = 100

# The base is the average of the base period, not one month of it. Using a single
# month would hand whatever happened that month to every number in the series.
base_2019 = cpi.loc[cpi.index.year == 2019, "cpi"].mean()

# the one line that does the work
cpi_rebased = cpi["cpi"] / base_2019 * 100

print(f"1982-84 = 100 basis : 2019 average is {base_2019:.2f}")
print(f"2019 = 100 basis    : 2019 average is {cpi_rebased[cpi_rebased.index.year == 2019].mean():.2f}")
print()
for y in (2019, 2022, 2024, 2026):
    raw = cpi.loc[cpi.index.year == y, "cpi"].mean()
    reb = cpi_rebased[cpi_rebased.index.year == y].mean()
    print(f"{y}  raw {raw:7.2f}   rebased {reb:6.2f}")


**Expected output:** the 2019 average is `100.00` on the new basis — that is the check that
you rebased to the year you meant to. The raw numbers do not change shape; only the scale
does. 2024 reads `122.70`, meaning prices were about **22.7% above their 2019 average**,
which is a sentence you can put in front of a client. `313.70` is not.

Note the 2026 row covers only the months FRED has published so far this year, so it is a
part-year average. Say so if you quote it.


## Part 3: The Fifty-Year Wage Divergence (GUIDED — 10 min)

Now apply `deflate_series()` to nominal wages and see the divergence.

In [ ]:
# GUIDED — run as-is
# Step 5: pull nominal wages and CPI from 1964, then deflate the wages
# AHETPI: Average Hourly Earnings of Production and Nonsupervisory Employees, Total Private

wages_nominal = load_fred("AHETPI", start="1964-01-01")
cpi_wages = load_fred("CPIAUCSL", start="1964-01-01")

wages_real = deflate_series(wages_nominal, cpi_wages, base_year=2020)

print(f"Nominal wage, first month to last: ${wages_nominal.iloc[0]:.2f} to ${wages_nominal.iloc[-1]:.2f}")
print(f"Real wage, first month to last:    ${wages_real.iloc[0]:.2f} to ${wages_real.iloc[-1]:.2f} (2020 $)")

**Expected output:** the nominal wage goes from \$2.50 an hour in January 1964 to
over \$32 now. In 2020 dollars, the same wage goes from about \$21 to about
\$25. These are the first and last months, not the lowest and highest: the
chart below shows what happens in between.

In [ ]:
# GUIDED — run as-is
# Step 6: plot the divergence, the key chart of the chapter

def plot_nominal_real(nominal, real, what, marker=""):
    """Draw a nominal series and its real (2020 $) version on one chart."""
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(nominal.index, nominal, marker=marker, label=f"Nominal {what}")
    ax.plot(real.index, real, marker=marker, label=f"Real {what} (2020 $)")
    ax.set_xlabel("Year")
    ax.legend()
    return ax


# Step 9 reuses this function for the Big Mac
ax = plot_nominal_real(wages_nominal, wages_real, "wage")
ax.set_ylabel("Average hourly earnings ($)")
ax.set_title("Since 1964, nominal pay rose thirteenfold and real pay about a fifth")
plt.show()

### Interpretation Question 2

Look at the chart above.

**Q1:** Real wages hit a high in the early 1970s. About when did they first return to it, and what happened around that high?

**Q2:** If you only looked at the nominal (blue) line, would you conclude that workers' purchasing power has been rising? Why is this misleading?

**Q3:** This chart is the "vibecession" in one picture. Explain in 2-3 sentences why consumers in 2023 felt worse off even though CPI inflation was falling.

*Your answers here:*



## Part 4: Deflating the Big Mac (YOUR TURN — 10 min)

Now apply what you have learned. Load the Big Mac data from Lab 1 and deflate US
prices to see how much of the price increase is "real" and how much is inflation.

Step 7 loads the data; run it as it is. Step 8 has three blanks, and each one
takes a single expression.

In [ ]:
# GUIDED — run as-is
# Step 7: load the Big Mac data (the same file as Lab 1)

url = "https://raw.githubusercontent.com/TheEconomist/big-mac-data/master/output-data/big-mac-full-index.csv"
bigmac = pd.read_csv(url, parse_dates=["date"])

# The US rows, with dates as the index. .copy() makes us its own table,
# so adding columns to it in Step 8 leaves bigmac alone
is_us = bigmac["name"] == "United States"
us = bigmac.loc[is_us, ["date", "dollar_price"]].copy()
us = us.set_index("date").sort_index()
us.columns = ["bigmac_nominal"]

print(f"US Big Mac observations: {len(us)}")
print(f"Date range: {us.index.min().date()} to {us.index.max().date()}")
print(f"Price range: ${us['bigmac_nominal'].min():.2f} to ${us['bigmac_nominal'].max():.2f}")

In [ ]:
# YOUR TASK — fill in the three blanks (____)
# Step 8: Deflate Big Mac prices to 2020 dollars

# Step A (done for you): CPI from 2000 onward, as a Series. No API key needed.
cpi_bm = load_fred("CPIAUCSL", start="2000-01-01")

# Step B (done for you): the Big Mac is priced once or twice a year, the CPI
# monthly. For each Big Mac date, asof takes the latest CPI on or before it.
us["cpi"] = cpi_bm.asof(us.index)          # asof fills a missing month with the one before

# Step C: the base-year CPI. Average every 2020 observation of cpi_bm.
#   Hint: Step 4 averaged one year of CPI. Its cpi was a DataFrame; cpi_bm is a Series.
base_cpi = ____

# Step D: deflate.  Real = (Nominal / CPI) * Base CPI
#   Hint: the nominal column is us["bigmac_nominal"] and the CPI column is
#   us["cpi"]. Whole columns divide element by element — no loop.
bigmac_real = ____

us["bigmac_real"] = bigmac_real          # done for you

# Step E: the percentage change from the first observation to the last, for
#   each series. .iloc[-1] is the last row, .iloc[0] the first.
nominal_change = ((us["bigmac_nominal"].iloc[-1] / us["bigmac_nominal"].iloc[0]) - 1) * 100
real_change = ____
cpi_change = ((us["cpi"].iloc[-1] / us["cpi"].iloc[0]) - 1) * 100   # done for you

print(f"Nominal change: {nominal_change:.0f}%")
print(f"Real change: {real_change:.0f}%")
print(f"CPI change over the same dates: {cpi_change:.0f}%")

In [ ]:
# CHECK — run as-is after Step 8
# assert stops the cell with the message if its condition is False

# 1. Units: in 2020, a price in 2020 dollars should match the nominal price (within 3%)
in_2020 = us[us.index.year == 2020]
ratio = (in_2020["bigmac_real"] / in_2020["bigmac_nominal"]).mean()
assert 0.97 < ratio < 1.03, (
    f"In 2020 your real price is {ratio:.3f} times the nominal price; it should be "
    "about 1. A result near 0.004 means Base CPI is missing; near 0.39, the base is "
    "100 instead of the 2020 CPI.")

# 2. Base year: base_cpi should be the 2020 average, not one month
months_2020 = cpi_bm[cpi_bm.index.year == 2020]
assert months_2020.min() <= base_cpi <= months_2020.max(), (
    f"base_cpi = {base_cpi:.2f} is not a 2020 CPI level.")
assert not (months_2020 == base_cpi).any(), (
    "base_cpi equals a single month's CPI; Step C asks for the 2020 average.")

# 3. Growth rates multiply: (1 + nominal) = (1 + real) x (1 + CPI)
assert abs((1 + nominal_change / 100) - (1 + real_change / 100) * (1 + cpi_change / 100)) < 0.01, (
    "real_change is inconsistent with nominal_change and the CPI: "
    "(1 + nominal) should equal (1 + real) x (1 + CPI).")

print("Step 8 check passed: real prices are in 2020 dollars.")
print(f"  (1 + {nominal_change:.0f}%) = (1 + {real_change:.0f}%) x (1 + {cpi_change:.0f}%)")
print(f"  nominal - real = {nominal_change - real_change:.0f} points; CPI growth = {cpi_change:.0f}%")

In [ ]:
# GUIDED — run as-is (after Step 8 is filled in)
# Step 9: the Step 6 chart, pointed at a burger

# marker="o" draws a dot at every observation
ax = plot_nominal_real(us["bigmac_nominal"], us["bigmac_real"], "Big Mac price", marker="o")
ax.set_ylabel("US Big Mac price ($)")
ax.set_title("Since 2000 the US Big Mac price nearly tripled; after inflation it rose about 40%")
plt.show()

**Try this:** the Big Mac is priced only once or twice a year, so Step 9 marks
each observation with a dot. Change `marker="o"` to `marker=""` and re-run. In
your answers below, add one sentence on which version is more honest about how
much data actually exists.

### Interpretation Question 3

**Q1:** Your check printed three growth rates: nominal, real, and the CPI. Why is
*nominal minus real* not equal to the CPI's growth? Use the identity the check
printed, and say which of the three numbers is "inflation".

**Q2:** In Chapter 1, you compared Big Mac prices *across countries* (PPP). In this lab, you compared Big Mac prices *across time* (deflation). Both are index operations. What is the key difference between them?

**Q3:** A CEO tells shareholders: "Our revenue grew 15% this year." Inflation was 5%. What question should you ask, and what is the real growth rate?

*Your answers here:*



---

## Part 5 (Optional — 10 min): The same data through the FRED API

Everything above used FRED's CSV link, which needs no credential. Larger
projects usually use the API instead, because it can search the catalogue, pull
metadata, and handle revisions — none of which a CSV link can do. That route
needs the API key you set up at the top of this notebook.

**If you did not get a key, skip this part.** Nothing later depends on it.

The first cell reads your key from Colab Secrets. If it finds none, it stops with
"No FRED API key found" and says what to do. Without that check, the request
would fail later with a long error from FRED that does not say what went wrong.

**One more piece of syntax.** The second cell starts with `!pip install fredapi -q`.
A line starting with `!` is **not Python** — it is a command sent to the operating
system of the machine Colab gave you. `pip` is Python's package installer,
`fredapi` is the package, and `-q` means "quietly". Colab hands you a fresh, empty
machine each session, so you run it once per session.

In [ ]:
# skip-run — needs your own FRED API key; the rest of the lab does not.
# OPTIONAL — Part 5, cell 1: load your key from Colab Secrets
# The key lives in Colab Secrets (key icon, left sidebar), so it is never in this file.

# try runs its block; if any line there fails, Python runs the except block instead
try:
    from google.colab import userdata                   # exists only inside Colab
    FRED_API_KEY = userdata.get("FRED_API_KEY")
except Exception:        # broad on purpose: no module, no secret or no access all mean "no key"
    import os
    FRED_API_KEY = os.environ.get("FRED_API_KEY", "")   # outside Colab: an environment variable

if not FRED_API_KEY:
    raise ValueError(
        "No FRED API key found. Nothing is broken; this cell has no key to send yet.\n"
        "  1. Get a free key: https://fredaccount.stlouisfed.org/apikeys\n"
        "  2. In Colab's left sidebar click the key icon (Secrets), then + Add new secret.\n"
        "  3. Name it exactly FRED_API_KEY, paste the key as the Value, and switch\n"
        "     Notebook access ON. Missing that switch is the most common cause of this message.\n"
        "  4. Re-run this cell.\n"
        "Or skip Part 5: Parts 1-4 need no key and are the graded content of this lab."
    )

print(f"Key loaded ({len(FRED_API_KEY)} characters). It is not printed and not saved in this notebook.")

In [ ]:
# skip-run — needs your own FRED API key.
# OPTIONAL — Part 5, cell 2: the same CPI series, via fredapi

# A shell command, not Python (see above): installs fredapi on this Colab machine
!pip install fredapi -q

from fredapi import Fred

fred = Fred(api_key=FRED_API_KEY)
cpi_api = fred.get_series("CPIAUCSL", observation_start="1947-01-01")

# get_series keeps a month with no value as NaN; load_fred drops it, so the counts can differ
print(f"Via the API:  {len(cpi_api)} observations, latest {cpi_api.iloc[-1]:.1f}")
print(f"Via the CSV:  {len(cpi)} observations, latest {cpi['cpi'].iloc[-1]:.1f}")

# Something a CSV link cannot do: search FRED's catalogue
print()
print(fred.search("unemployment rate", limit=3)[["title", "frequency"]].to_string())

### Interpretation Question 4

**Q1:** Parts 1-4 pulled the same CPI series with no credential at all. Why does
FRED bother issuing API keys, if the data is free either way?

**Q2:** You are about to push this notebook to a public GitHub repository, and your
key is nowhere in it. Explain the mechanism that makes that true — and say what you
would have to do if a key did reach a public repo.

*Your answers here:*

1. 
2. 

---
## AI-Assisted Expansion: Interactive Deflation Explorer

**AI policy: foundations first, expansion second.** You have pulled CPI from FRED, deflated nominal series, and read real against nominal charts by hand. From here on, AI may write code for you (the Co-Pilot Rule).

### Your Expansion Task
Build an interactive explorer (ipywidgets + matplotlib, running inside Colab) that lets the user:
1. Choose a series — average hourly earnings or the US Big Mac price
2. Move a base-year slider and watch the real series relabel into that year's dollars
3. Toggle between nominal and real
4. Read the growth rate over the chosen window, which should NOT change with the base year

### P.R.I.M.E. Prompt
Copy and paste this into Claude or ChatGPT:

In [ ]:
# AI EXPANSION — copy the prompt below into Claude, then paste
# its code at the bottom of this cell. Run it and check it.

# [Prep] Act as an expert Python data scientist who teaches
# introductory economics with interactive notebooks.
#
# [Request] I just finished a lab where I pulled CPI from FRED
# with a load_fred() helper, deflated nominal wages with
# deflate_series(nominal, cpi, base_year), and deflated US Big
# Mac prices to 2020 dollars. Build an interactive explorer that
# runs inside Google Colab:
#   1. a dropdown to pick average hourly earnings
#      (wages_nominal, deflated with cpi_wages) or the
#      US Big Mac price (us["bigmac_nominal"])
#   2. a base-year slider (2000-2025) that re-deflates the series
#   3. a nominal / real toggle
#   4. a text readout of the percentage change from the first to
#      the last observation, in both nominal and real terms
#
# [Iterate] Use ipywidgets and matplotlib only. Reuse my
# variables and functions: load_fred, deflate_series, cpi_bm,
# cpi_wages, wages_nominal, wages_real, us. Do not use deprecated pandas functions.
#
# [Mechanism Check] Add inline comments explaining:
#   - why moving the base year changes the dollar LEVEL of the
#     real series but not its growth rate
#   - why the Big Mac series needs .asof() to line up with the
#     monthly CPI
#   - how the widget callback redraws the chart
#
# [Evaluate] Explain what the explorer shows about the claim
# "real wages have been flat for fifty years".

# PASTE AI-GENERATED CODE BELOW:

### Document the interaction

Three short answers. This is what is graded for the expansion, not the code.

1. **What did the AI get wrong or leave out?** Name one line you had to change,
   or one claim in its explanation that your own output contradicts.
2. **How did you verify it?** Point to a number the dashboard shows that you
   also computed by hand earlier in this lab, and say whether they match.
3. **What would you ask differently next time?**

*Your answers here:*

1.
2.
3.

## Part 6: Write your README

Your README is what a reader sees before they open a single cell. Draft it with
the README prompt in the Digital Portfolio section below, then check every
number in the draft against your own output.

---
## Digital Portfolio: README Prompt

### Draft your README
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.** Replace every `[YOUR VALUE]` with the number from your own output before you paste it.

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Pulled CPI and average hourly earnings from FRED with no API key
* Used deflate_series() to convert wages to constant 2020 dollars
* Showed nominal hourly earnings rose from $[YOUR VALUE] to $[YOUR VALUE]
  while real earnings moved from $[YOUR VALUE] to $[YOUR VALUE]
* Deflated the US Big Mac price: nominal +[YOUR VALUE]%, real
  +[YOUR VALUE]%, CPI +[YOUR VALUE]% over the same dates
* Built an interactive deflation explorer with a base-year slider

**Please write a README.md entry including:**
1. Project Title: Deflating Economic Data — Nominal vs. Real
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab02-deflation`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab02-deflation`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab02-deflation`, branch `main`,
   commit message `Lab 02 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.

---

**End of Lab 2.** You pulled CPI and wages from FRED, used `deflate_series()` to
put wages since 1964 in 2020 dollars, and split the Big Mac's price rise into
inflation and a real increase. Never compare money across time without deflating
first.